# Chapter 16 — Search, Memory and Long Context

**Book alignment:** DSPy From First Principles, Chapter 16

**Question this notebook isolates:** Does the frozen v1 metric assign zero `changed` credit to a punctuation-only rewrite while giving full credit to a lexical contraction?


In [ ]:
from pathlib import Path
import random
import sys

random.seed(0)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import dspy
from common.data import canonical_split
from common.fingerprints import fingerprint
from common.metrics import changed_signal, score_editorial_output
from common.retrieval import (
    EvidencePacket,
    decision_time_case_memory,
    fixed_lexical_retriever,
    labeled_training_memory,
)


class RewriteWithEvidence(dspy.Signature):
    """Rewrite one sentence using only the task inputs and supplied evidence."""

    sentence: str = dspy.InputField()
    goal: str = dspy.InputField()
    context: str = dspy.InputField()
    evidence: str = dspy.InputField()
    rewritten_text: str = dspy.OutputField()


rewrite_program = dspy.Predict(RewriteWithEvidence)
print("dspy", dspy.__version__, "| rewrite program constructed, never executed")


## Evidence role is not evidence quality

One target (`ed-003`), one fixed program. Decision-time case memory over train-only cases is the negative control: it must select nothing, while labeled training memory carries a richer, non-equivalent role.


In [ ]:
split = canonical_split()
target = next(c for c in split.dev if c.case_id == "ed-003")
train = list(split.train)

mem = decision_time_case_memory(target, train, limit=2)
fixed = fixed_lexical_retriever(target.goal, train, limit=2)
labeled = labeled_training_memory(target.goal, train, limit=2)
fixed_again = fixed_lexical_retriever(target.goal, train, limit=2)

print("target:", target.case_id, "|", target.source_group)
print("decision-time memory selected:", [i.case_id for i in mem.selected])
print("fixed lexical selected:     ", [i.case_id for i in fixed.selected])
print("labeled memory selected:    ", [i.case_id for i in labeled.selected], "(role=labeled_training_only)")
print("fixed fingerprint stable:", fixed.to_record()["packet_fingerprint"] == fixed_again.to_record()["packet_fingerprint"])


In [ ]:
assert target.case_id == "ed-003" and target.case_id in set(split.dev_ids)
assert len(mem.selected) == 0
assert [i.case_id for i in labeled.selected] == ["ed-001", "ed-002"]
assert [i.case_id for i in fixed.selected] == ["ed-001", "ed-002"]
assert all(i.case_id in set(split.train_ids) for i in labeled.selected)
print("negative control holds: no train-only case shares the dialogue group")


## A punctuation-only edit gets the 0.65 floor

The `changed` component compares normalized lexical tokens and discards punctuation, so adding quotation marks earns nothing while contracting `do not` to `don't` earns full credit.


In [ ]:
punct_only = '"I do not think we should go there," Anna said, "because it is unsafe."'
contraction = "I don't think we should go there, Anna said, because it's unsafe."
b_punct = score_editorial_output(target, punct_only)
b_contr = score_editorial_output(target, contraction)
print(f"punctuation-only: changed={b_punct.changed_score} scope={b_punct.scope_length_score} overlap={b_punct.reference_overlap_score} -> {b_punct.score:.4f}")
print(f"contraction:      changed={b_contr.changed_score} scope={b_contr.scope_length_score} overlap={b_contr.reference_overlap_score:.4f} -> {b_contr.score:.4f}")


In [ ]:
assert changed_signal(target, punct_only) == 0.0
assert changed_signal(target, contraction) == 1.0
assert abs(b_punct.score - 0.65) < 1e-9
assert abs(b_contr.score - 0.8769230769230769) < 1e-9
assert b_punct.score < b_contr.score
print("metric defect reproduced: goal-satisfying punctuation ranks below a contraction")


## Fallback score is not mechanism score

A selection step that crashes must record failure, and its downstream no-evidence rewrite must not be credited to the failed mechanism. Packets carry role, status, and fingerprint so the comparison stays auditable.


In [ ]:
direct = EvidencePacket("direct_context", target.case_id, ())
fallback = EvidencePacket("rlm_guideline_exploration", target.case_id, ())
direct_rec, fallback_rec = direct.to_record(), fallback.to_record()
failed_selection = {
    "policy": "rlm_guideline_exploration",
    "selection_status": "failed",
    "selected_evidence_ids": [],
    "evidence_role": "decision_time_guidelines",
}
print("direct evidence chars:", direct_rec["evidence_size_chars"])
print("fallback evidence chars:", fallback_rec["evidence_size_chars"])
print("labeled role flagged:", "labeled_training_only" != "decision_time_guidelines")


In [ ]:
assert direct_rec["evidence_size_chars"] == 0 == fallback_rec["evidence_size_chars"]
assert direct_rec["packet_fingerprint"] != labeled.to_record()["packet_fingerprint"]
assert failed_selection["selection_status"] == "failed"
assert failed_selection["selected_evidence_ids"] == []
print("fallback recorded as failure with empty evidence, not as an RLM result")


## What we earned

Evidence selection is program behavior with its own cost and provenance, and the evaluator is a separate instrument whose `changed` defect dominates the apparent ranking.

Notebook 17 / Chapter 17 attacks the failure no retriever fixes: committing to one reasoning path instead of searching several.
